# 🌿 Reconocimiento de hojas de plantas con una Red Neuronal Convolucional (CNN)

**Objetivo:** entrenar una RNA convolucional que reconozca **185 especies de árboles** a partir de la foto de una hoja, y exportarla para usarla en una página web (`index.html`) que toma la foto directamente con el celular.

**Dataset público:** [LeafSnap](http://leafsnap.com/dataset/) (Kumar et al., ECCV 2012): ~31.000 imágenes de 185 especies, con fotos de laboratorio (*lab*) y fotos de campo tomadas con celular (*field*).

**Técnica:** *Transfer learning* con **MobileNetV2** (una CNN preentrenada en ImageNet y diseñada para celulares) en dos fases:
1. **Extracción de características:** se congelan las capas convolucionales y se entrena solo el clasificador nuevo.
2. **Fine-tuning:** se descongelan las últimas capas convolucionales y se ajustan con una tasa de aprendizaje muy baja.

**Flujo:** Datos → Preprocesamiento y aumento → CNN → Evaluación → Exportación a TensorFlow.js → Página web.

> ⚙️ Antes de empezar: **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)**.

## 1. Configuración

In [ ]:
import os, sys, json, random, shutil, subprocess, pathlib, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from keras import layers

print("TensorFlow:", tf.__version__, "| Keras:", keras.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "⚠️ No hay GPU (activa la T4 en Colab)")

# ---- Parámetros del experimento (puedes cambiarlos) ----
IMG_SIZE = 224            # tamaño de entrada de la CNN
BATCH_SIZE = 64
EPOCHS_FASE1 = 8          # clasificador con la base congelada
EPOCHS_FASE2 = 12         # fine-tuning
CAPAS_A_DESCONGELAR = 40  # últimas capas de MobileNetV2 que se reentrenan
MIN_IMAGENES = 20         # se descartan especies con menos imágenes
MAX_CLASES = None         # None = todas las especies (185). Ej.: 60 para entrenar más rápido
SEED = 42

TEST_MODE = os.environ.get("TEST_MODE") == "1"   # solo para pruebas internas
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

## 2. Descarga del dataset LeafSnap
Se descarga desde el espejo público de Kaggle con `kagglehub` (no necesita cuenta). Si falla, se usa el enlace oficial de leafsnap.com.

In [ ]:
def descargar_leafsnap():
    try:
        import kagglehub
    except ImportError:
        subprocess.run(["pip", "install", "-q", "kagglehub"], check=True)
        import kagglehub
    try:
        ruta = kagglehub.dataset_download("xhlulu/leafsnap-dataset")
        print("Descargado desde Kaggle en:", ruta)
        return pathlib.Path(ruta)
    except Exception as e:
        print("Kaggle no disponible (", e, ") → probando el enlace oficial…")
    destino = pathlib.Path("/content/leafsnap")
    destino.mkdir(parents=True, exist_ok=True)
    tar = destino / "leafsnap-dataset.tar"
    subprocess.run(["wget", "-q", "-O", str(tar),
                    "http://leafsnap.com/static/dataset/leafsnap-dataset.tar"], check=True)
    subprocess.run(["tar", "-xf", str(tar), "-C", str(destino)], check=True)
    tar.unlink()
    return destino

if TEST_MODE:
    DATA_DIR = pathlib.Path(os.environ["TEST_DATA_DIR"])
else:
    DATA_DIR = descargar_leafsnap()

## 3. Exploración de los datos
Cada imagen está en `images/<lab|field>/<especie>/archivo.jpg`. Se ignoran las imágenes *segmentadas* (máscaras en blanco y negro).

In [ ]:
EXT = {".jpg", ".jpeg", ".png"}
registros = []
for p in DATA_DIR.rglob("*"):
    if p.suffix.lower() not in EXT:
        continue
    partes = [s.lower() for s in p.parts]
    if any("segment" in s for s in partes):
        continue
    origen = "field" if "field" in partes else ("lab" if "lab" in partes else "otro")
    registros.append({"ruta": str(p), "especie": p.parent.name.lower(), "origen": origen})

df = pd.DataFrame(registros).drop_duplicates("ruta")
conteo = df["especie"].value_counts()
especies = conteo[conteo >= MIN_IMAGENES].index.tolist()
if MAX_CLASES:
    especies = especies[:MAX_CLASES]
df = df[df["especie"].isin(especies)].reset_index(drop=True)

CLASES = sorted(especies)
NUM_CLASES = len(CLASES)
clase_a_id = {c: i for i, c in enumerate(CLASES)}
df["label"] = df["especie"].map(clase_a_id)

print(f"Imágenes: {len(df):,} | Especies: {NUM_CLASES}")
print(df["origen"].value_counts().to_string())
assert NUM_CLASES >= 50 or TEST_MODE, "El trabajo exige al menos 50 tipos de hojas"

fig, ax = plt.subplots(figsize=(14, 4))
df["especie"].value_counts().plot(kind="bar", ax=ax, color="#2e7d32")
ax.set_title("Imágenes por especie"); ax.set_xticks([]); ax.set_xlabel(f"{NUM_CLASES} especies")
plt.show()

In [ ]:
# Muestra visual de algunas especies
muestra = df.groupby("especie").sample(1, random_state=SEED).sample(min(12, NUM_CLASES), random_state=SEED)
fig, axes = plt.subplots(2, 6, figsize=(16, 6))
for ax, (_, fila) in zip(axes.ravel(), muestra.iterrows()):
    ax.imshow(plt.imread(fila["ruta"])); ax.axis("off")
    ax.set_title(fila["especie"].replace("_", " ").capitalize() + f"\n({fila['origen']})", fontsize=9)
plt.tight_layout(); plt.show()

## 4. División entrenamiento / validación / prueba (80 / 10 / 10)
Estratificada por especie, para que todas las especies aparezcan en los tres conjuntos.

In [ ]:
from sklearn.model_selection import train_test_split
df_train, df_tmp = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=SEED)
df_val, df_test = train_test_split(df_tmp, test_size=0.5, stratify=df_tmp["label"], random_state=SEED)
print(f"Entrenamiento: {len(df_train):,} | Validación: {len(df_val):,} | Prueba: {len(df_test):,}")

## 5. Preprocesamiento y aumento de datos
- **Recorte central cuadrado + redimensión a 224×224.** La página web hace exactamente lo mismo con la foto del celular.
- **Aumento de datos** (solo en entrenamiento): rotaciones, volteos, zoom, brillo y contraste. Simula fotos reales tomadas en distintos ángulos y luces.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def cargar_imagen(ruta, label):
    img = tf.io.decode_image(tf.io.read_file(ruta), channels=3, expand_animations=False)
    h, w = tf.shape(img)[0], tf.shape(img)[1]
    lado = tf.minimum(h, w)
    img = tf.image.crop_to_bounding_box(img, (h - lado) // 2, (w - lado) // 2, lado, lado)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE), antialias=True)
    return tf.cast(img, tf.float32), label   # valores 0-255; la normalización va dentro del modelo

aumento = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.25, fill_mode="reflect"),
    layers.RandomZoom((-0.2, 0.1), fill_mode="reflect"),
    layers.RandomTranslation(0.1, 0.1, fill_mode="reflect"),
    layers.RandomBrightness(0.2, value_range=(0, 255)),
    layers.RandomContrast(0.2),
], name="aumento_datos")

def crear_dataset(d, entrenamiento=False):
    ds = tf.data.Dataset.from_tensor_slices((d["ruta"].values, d["label"].values))
    if entrenamiento:
        ds = ds.shuffle(len(d), seed=SEED)
    ds = ds.map(cargar_imagen, num_parallel_calls=AUTOTUNE)
    if not entrenamiento:
        ds = ds.cache()
    ds = ds.batch(BATCH_SIZE)
    if entrenamiento:
        ds = ds.map(lambda x, y: (tf.clip_by_value(aumento(x, training=True), 0, 255), y),
                    num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

ds_train = crear_dataset(df_train, entrenamiento=True)
ds_val = crear_dataset(df_val)
ds_test = crear_dataset(df_test)

# Visualizar el aumento de datos sobre una misma hoja
x0, _ = cargar_imagen(df_train["ruta"].iloc[0], 0)
fig, axes = plt.subplots(1, 6, figsize=(16, 3))
for ax in axes:
    ax.imshow(tf.clip_by_value(aumento(x0[None], training=True)[0], 0, 255).numpy().astype("uint8")); ax.axis("off")
plt.suptitle("Aumento de datos: la misma hoja, distintas variaciones"); plt.show()

## 6. Arquitectura de la CNN

```
Imagen 224×224×3 (0-255)
  → Rescaling a [-1, 1]
  → MobileNetV2 (bloques convolucionales: Conv 3×3, Depthwise Conv, BatchNorm, ReLU6, residuales invertidos)
  → GlobalAveragePooling2D (mapa 7×7×1280 → vector de 1280)
  → Dropout 0.3
  → Dense(185, softmax)  → probabilidad de cada especie
```

In [ ]:
base = keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False,
    weights=None if TEST_MODE else "imagenet")
base.trainable = False

entradas = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="imagen")
x = layers.Rescaling(1 / 127.5, offset=-1, name="normalizacion")(entradas)
x = base(x, training=False)          # BatchNorm siempre en modo inferencia
x = layers.GlobalAveragePooling2D(name="pooling_global")(x)
x = layers.Dropout(0.3, name="dropout")(x)
salidas = layers.Dense(NUM_CLASES, activation="softmax", name="especie")(x)
modelo = keras.Model(entradas, salidas, name="cnn_hojas")
modelo.summary()

## 7. Fase 1: entrenar solo el clasificador

In [ ]:
metricas = ["accuracy", keras.metrics.SparseTopKCategoricalAccuracy(k=5, name="top5")]
modelo.compile(optimizer=keras.optimizers.Adam(1e-3),
               loss=keras.losses.SparseCategoricalCrossentropy(),
               metrics=metricas)

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=4, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2),
]
hist1 = modelo.fit(ds_train, validation_data=ds_val, epochs=EPOCHS_FASE1, callbacks=callbacks)

## 8. Fase 2: fine-tuning de las últimas capas convolucionales

In [ ]:
base.trainable = True
for capa in base.layers[:-CAPAS_A_DESCONGELAR]:
    capa.trainable = False
for capa in base.layers:                       # las BatchNorm se mantienen congeladas
    if isinstance(capa, layers.BatchNormalization):
        capa.trainable = False

modelo.compile(optimizer=keras.optimizers.Adam(1e-5),
               loss=keras.losses.SparseCategoricalCrossentropy(),
               metrics=metricas)
hist2 = modelo.fit(ds_train, validation_data=ds_val, epochs=EPOCHS_FASE2, callbacks=callbacks)

## 9. Curvas de aprendizaje

In [ ]:
def unir(h1, h2, k):
    return h1.history[k] + h2.history[k]

n1 = len(hist1.history["loss"])
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, k, titulo in [(axes[0], "accuracy", "Precisión"), (axes[1], "loss", "Pérdida")]:
    ax.plot(unir(hist1, hist2, k), label="entrenamiento")
    ax.plot(unir(hist1, hist2, "val_" + k), label="validación")
    ax.axvline(n1 - 0.5, color="gray", ls="--", label="inicio fine-tuning")
    ax.set_title(titulo); ax.set_xlabel("época"); ax.legend()
plt.show()

## 10. Evaluación en el conjunto de prueba (imágenes nunca vistas)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

resultado = modelo.evaluate(ds_test, return_dict=True)
print(f"\n✅ Precisión top-1: {resultado['accuracy']:.2%} | top-5: {resultado['top5']:.2%}")

y_true = df_test["label"].values
y_prob = modelo.predict(ds_test)
y_pred = y_prob.argmax(1)

origen_test = df_test["origen"].values
for o in ["lab", "field"]:
    m = origen_test == o
    if m.any():
        print(f"   Precisión en fotos '{o}': {(y_pred[m] == y_true[m]).mean():.2%} ({m.sum()} imágenes)")

reporte = pd.DataFrame(classification_report(
    y_true, y_pred, labels=range(NUM_CLASES), target_names=CLASES,
    output_dict=True, zero_division=0)).T.iloc[:NUM_CLASES]
print("\nEspecies más difíciles:")
print(reporte.sort_values("f1-score").head(10)[["precision", "recall", "f1-score", "support"]])

cm = confusion_matrix(y_true, y_pred, labels=range(NUM_CLASES), normalize="true")
plt.figure(figsize=(9, 8))
plt.imshow(cm, cmap="Greens"); plt.colorbar()
plt.title("Matriz de confusión normalizada (diagonal = aciertos)")
plt.xlabel("predicción"); plt.ylabel("real"); plt.show()

## 11. Probar con tu propia foto
Sube una foto de una hoja (por ejemplo, tomada con tu celular).

In [ ]:
def predecir_archivo(ruta, k=5):
    x, _ = cargar_imagen(ruta, 0)
    p = modelo.predict(x[None], verbose=0)[0]
    plt.imshow(x.numpy().astype("uint8")); plt.axis("off"); plt.show()
    for i in p.argsort()[::-1][:k]:
        print(f"{CLASES[i].replace('_', ' ').capitalize():35s} {p[i]:.1%}")

try:
    from google.colab import files
    for nombre in files.upload():
        predecir_archivo(nombre)
except Exception:
    predecir_archivo(df_test["ruta"].iloc[0])

## 12. Exportar a TensorFlow.js (para la página web)
Se genera la carpeta `modelo_web/` con:
- `model.json` + `group1-shard*.bin`: la CNN en formato web (cuantizada a float16 para que pese la mitad).
- `labels.json`: nombres de las especies y métricas del modelo.

In [ ]:
SAVED = "saved_model_hojas"
WEB = pathlib.Path("modelo_web")
shutil.rmtree(SAVED, ignore_errors=True); shutil.rmtree(WEB, ignore_errors=True)
modelo.export(SAVED, verbose=False)

# Se instala el conversor SIN sus dependencias pesadas (jax, decision-forests), porque estas
# reinstalan otra versión de TensorFlow y dañan el entorno de Colab.
if not TEST_MODE:
    tf_ver = ".".join(tf.__version__.split(".")[:2])
    subprocess.run(["pip", "install", "-q", "--no-deps", "tensorflowjs==4.22.0",
                    "tensorflow-hub>=0.16.1"], check=True)
    try:
        import tf_keras  # noqa: F401
    except ImportError:
        subprocess.run(["pip", "install", "-q", "--no-deps", f"tf_keras=={tf_ver}.*"], check=True)

pathlib.Path("convertir.py").write_text('''
import sys, types, importlib, warnings, numpy as np
warnings.filterwarnings("ignore")
for a, b in [("object", object), ("bool", bool)]:
    if a not in np.__dict__: setattr(np, a, b)
def stub(nombre, **attrs):
    try:
        importlib.import_module(nombre)
    except Exception:
        m = types.ModuleType(nombre); m.__dict__.update(attrs); sys.modules[nombre] = m
stub("tensorflow_decision_forests")
stub("tensorflowjs.converters.jax_conversion", convert_jax=None)
from tensorflowjs.converters.converter import pip_main
sys.argv[0] = "tensorflowjs_converter"
pip_main()
''')
r = subprocess.run([sys.executable, "convertir.py",
                    "--input_format=tf_saved_model", "--output_format=tfjs_graph_model",
                    "--signature_name=serving_default", "--quantize_float16=*",
                    SAVED, str(WEB)], capture_output=True, text=True)
if r.returncode != 0:
    print(r.stderr[-3000:]); raise RuntimeError("Falló la conversión a TensorFlow.js")

info = {
    "clases": CLASES,
    "nombres": [c.replace("_", " ").capitalize() for c in CLASES],
    "img_size": IMG_SIZE,
    "precision_test": round(float(resultado["accuracy"]), 4),
    "top5_test": round(float(resultado["top5"]), 4),
    "dataset": "LeafSnap (Kumar et al., 2012)",
    "arquitectura": "MobileNetV2 + transfer learning",
}
(WEB / "labels.json").write_text(json.dumps(info, ensure_ascii=False, indent=1))
print("Archivos:", sorted(p.name for p in WEB.iterdir()))
print(f"Tamaño total: {sum(p.stat().st_size for p in WEB.iterdir()) / 1e6:.1f} MB")

shutil.make_archive("modelo_web", "zip", WEB)
modelo.save("cnn_hojas.keras")   # copia del modelo completo, por si se necesita reentrenar
try:
    from google.colab import files
    files.download("modelo_web.zip")
except Exception:
    print("modelo_web.zip listo")

## 13. Siguiente paso
1. Descomprime `modelo_web.zip` dentro de la carpeta `model/` del repositorio de GitHub, junto a `index.html`.
2. Sube el repositorio y activa **GitHub Pages** (Settings → Pages → rama `main`).
3. Abre el enlace desde el celular, toma la foto de una hoja y ve la predicción.